# First Take

There is a class imbalance problem.\
That is, there are more records of one class than another.\
Strategies should be adopted to deal with this problem.

# Objective

Test different techniques to overcome the class imbalance problem.

Once this is done, random_forest will be used to classify the data, based on data from industrial sensors, which will classify each observation as:

- 0: (machine) does not need maintenance.
- 1: (machine) needs maintenance.

# Imports

In [ ]:
import imblearn
imblearn.__version__

In [ ]:
# Imports
import numpy as np
import pandas as pd

from imblearn.over_sampling import SMOTE

import lightgbm as lgb

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

import warnings
warnings.filterwarnings('ignore')

# Load Data

In [ ]:
file_path = 'data/dataset.csv'

In [ ]:
df = pd.read_csv(filepath_or_buffer = file_path)
df.head()

In [ ]:
df.info()

In [ ]:
df['manutencao_necessaria'].value_counts()

In [ ]:
class_labels = {0: 'does not need maintenance',
                1: 'needs maintenance'}

# Split Data

In [ ]:
X = df.drop('manutencao_necessaria', axis = 1)
y = df['manutencao_necessaria']

In [ ]:
X.shape, y.shape

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Pre-Processing

## Option 1: Class Weight

In [ ]:
scaler_v1 = StandardScaler()
scaler_v1.fit(X_train)

In [ ]:
X_train_scaled_v1 = scaler_v1.transform(X_train)
X_test_scaled_v1 = scaler_v1.transform(X_test)

In [ ]:
random_forest_v1 = RandomForestClassifier(n_estimators = 100,
                                          random_state = 42,
                                          class_weight = 'balanced')

In [ ]:
random_forest_v1.fit(X_train_scaled_v1, y_train)

In [ ]:
y_pred = random_forest_v1.predict(X_test_scaled_v1)
y_pred_proba = random_forest_v1.predict_proba(X_test_scaled_v1)[:,1]

In [ ]:
print(f'accuracy: {accuracy_score(y_test, y_pred):.3f}')
print(f'roc_auc: {roc_auc_score(y_test, y_pred_proba):.3f}')
print(f'classification_report:\n'
      f'{classification_report(y_test, y_pred)}')

## Option 2: Undersampling

In [ ]:
train_data_undersample = np.c_[X_train, y_train]

In [ ]:
df_under_class_0 = train_data_undersample[train_data_undersample[:,-1] == 0]
df_under_class_1 = train_data_undersample[train_data_undersample[:,-1] == 1]

In [ ]:
df_under_class_0.shape, df_under_class_1.shape

In [ ]:
# Undersampling com a função resample do sklearn
df_under_class_1_undersampled = resample(df_under_class_1,
                                    replace = False,
                                    random_state=42,
                                    n_samples = len(df_under_class_0))
df_under_class_1_undersampled.shape

In [ ]:
# Concatena
train_data_undersampled = np.r_[df_under_class_0, df_under_class_1_undersampled]
train_data_undersampled.shape

In [ ]:
X_train_undersampled = train_data_undersampled[:,:-1]
y_train_undersampled = train_data_undersampled[:,-1]

In [ ]:
scaler_v2 = StandardScaler()
scaler_v2.fit(X_train_undersampled)

In [ ]:
X_train_undersampled_scaled_v2 = scaler_v2.transform(X_train_undersampled)
X_test_scaled_v2 = scaler_v2.transform(X_test)

In [ ]:
random_forest_v2 = RandomForestClassifier(n_estimators = 100,
                                          random_state = 42)

In [ ]:
random_forest_v2.fit(X_train_undersampled_scaled_v2, y_train_undersampled)

In [ ]:
y_pred_v2 = random_forest_v1.predict(X_test_scaled_v2)
y_pred_proba_v2 = random_forest_v1.predict_proba(X_test_scaled_v2)[:,1]

In [ ]:
print(f'accuracy: {accuracy_score(y_test, y_pred_v2):.3f}')
print(f'roc_auc: {roc_auc_score(y_test, y_pred_proba_v2):.3f}')
print(f'classification_report:\n'
      f'{classification_report(y_test, y_pred_v2)}')

## Option 3: Oversampling

In [ ]:
train_data_oversample = np.c_[X_train, y_train]

In [ ]:
df_over_class_0 = train_data_oversample[train_data_oversample[:,-1] == 0]
df_over_class_1 = train_data_oversample[train_data_oversample[:,-1] == 1]

In [ ]:
df_over_class_0.shape, df_over_class_1.shape

In [ ]:
df_over_class_0_oversampled = resample(df_over_class_0,
                                    replace = True,
                                    random_state=42,
                                    n_samples = len(df_over_class_1))
df_over_class_0_oversampled.shape

In [ ]:
train_data_oversampled = np.r_[df_over_class_0_oversampled, df_over_class_1]
train_data_oversampled.shape

In [ ]:
X_train_oversampled = train_data_oversampled[:,:-1]
y_train_oversampled = train_data_oversampled[:,-1]

In [ ]:
scaler_v3 = StandardScaler()
scaler_v3.fit(X_train_oversampled)

In [ ]:
X_train_oversampled_scaled_v3 = scaler_v2.transform(X_train_oversampled)
X_test_scaled_v3 = scaler_v3.transform(X_test)

In [ ]:
random_forest_v3 = RandomForestClassifier(n_estimators = 100,
                                          random_state = 42)

In [ ]:
random_forest_v3.fit(X_train_oversampled_scaled_v3, y_train_oversampled)

In [ ]:
y_pred_v3 = random_forest_v3.predict(X_test_scaled_v3)
y_pred_proba_v3 = random_forest_v3.predict_proba(X_test_scaled_v3)[:,1]

In [ ]:
print(f'accuracy: {accuracy_score(y_test, y_pred_v3):.3f}')
print(f'roc_auc: {roc_auc_score(y_test, y_pred_proba_v3):.3f}')
print(f'classification_report:\n'
      f'{classification_report(y_test, y_pred_v3)}')

## Option 4: Smote Interpolation

- SMOTE (Synthetic Minority Over-sampling Technique)

In [ ]:
scaler_v4 = StandardScaler()
scaler_v4.fit(X_train)

In [ ]:
X_train_scaled_v4 = scaler_v4.transform(X_train)
X_test_scaled_v4 = scaler_v4.transform(X_test)

In [ ]:
smote_v1 = SMOTE(random_state=42)

In [ ]:
X_train_smote, y_train_smote = smote_v1.fit_resample(X_train_scaled_v4, y_train)

In [ ]:
y_train.value_counts()[1] * 2

In [ ]:
y_train_smote.shape

In [ ]:
random_forest_v4 = RandomForestClassifier(n_estimators = 100,
                                          random_state = 42)

In [ ]:
random_forest_v4.fit(X_train_smote, y_train_smote)

In [ ]:
y_pred_v4 = random_forest_v4.predict(X_test_scaled_v4)
y_pred_proba_v4 = random_forest_v4.predict_proba(X_test_scaled_v4)[:,1]

In [ ]:
print(f'accuracy: {accuracy_score(y_test, y_pred_v4):.3f}')
print(f'roc_auc: {roc_auc_score(y_test, y_pred_proba_v4):.3f}')
print(f'classification_report:\n'
      f'{classification_report(y_test, y_pred_v4)}')

# Model Selection

The best model is the version 1 because it performs as well as the others, and at the same time, it did not alter the data (it did not perform resampling).

The change was in the decision tree algorithm itself, where a proportional weight was added to each class.

# Inference

In [ ]:
def infer(data, labels, scaler, model):
    X_scaled = scaler.transform(data)
    y_pred = model.predict(X_scaled).item()
    return f'The machine {labels[y_pred]}.'

In [ ]:
df.columns

## Sample_1

In [ ]:
sample_1 = np.array([[0.5, 80, 102, 45, 8000]])

In [ ]:
infer(sample_1, class_labels, scaler_v1, random_forest_v1)

## Sample_2

In [ ]:
sample_2 = np.array([[0.89, 92, 96, 70, 600]])

In [ ]:
infer(sample_2, class_labels, scaler_v1, random_forest_v1)

# END